<a href="https://colab.research.google.com/github/c-marq/AI-Thinking-CAI1001C/blob/main/07-Classification-Part-1/Guided-Project/GP07_Heart_Disease_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GP07: Heart Disease Classification — k-NN & Decision Trees

**CAI1001C: Artificial Intelligence Thinking | Miami Dade College**
**Chapter 7: Classification Part 1**

---

**Guided Project — Reference Material (Not Graded)**

This notebook is your in-class reference. Follow along with your instructor, experiment in the "Your Turn" cells, and keep this notebook for future reference. All code runs top-to-bottom.

## Learning Objectives

By the end of this guided project, you will be able to:

- Train a k-Nearest Neighbors (k-NN) classifier on real patient data
- Experiment with different values of K and observe how accuracy changes
- Train a decision tree classifier and visualize its decision logic
- Compare two classification algorithms on the same dataset
- Interpret a classification report (precision, recall, accuracy)

## Setup

In [ ]:
# Run this cell first — installs and imports

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print('✅ All libraries loaded successfully!')

## Load the Heart Disease Dataset

We're working with a real-world dataset of **302 patients**. Each patient has 7 features (age, sex, chest pain type, resting blood pressure, cholesterol, max heart rate, exercise-induced angina) and one label: heart disease (1) or no heart disease (0).

In [ ]:
# Load the heart disease dataset from GitHub
url = "https://raw.githubusercontent.com/c-marq/AI-Thinking-CAI1001C/refs/heads/main/07-Classification-Part-1/Datasets/heart_disease_patients.csv"
df = pd.read_csv(url)

# Quick look at the data
print(f"Dataset: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"\nTarget distribution:")
print(df['heart_disease'].value_counts())
print(f"\nFirst 5 rows:")
df.head()

## Prepare the Data

We separate the features (X) from the label (y), then split into training (80%) and testing (20%) sets — the same workflow from Chapter 6. We also scale the features for k-NN.

In [ ]:
# Separate features and label
X = df.drop(columns='heart_disease')
y = df['heart_disease']

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Training set: {X_train.shape[0]} patients")
print(f"Testing set: {X_test.shape[0]} patients")

# Scale features for k-NN (decision trees don't need this)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

---
## Example 7.1: Your First k-NN Classifier

k-Nearest Neighbors classifies a new patient by finding the K most similar patients in the training data and taking a majority vote. We'll start with K=5 — asking the 5 closest neighbors.

In [ ]:
# ============================================
# Example 7.1: k-NN Heart Disease Classifier
# Purpose: Train k-NN with K=5, evaluate accuracy
# ============================================

# Train k-NN with K=5
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)

# Predict and evaluate
predictions = knn.predict(X_test_scaled)
accuracy = accuracy_score(y_test, predictions)
print(f"k-NN Accuracy (K=5): {accuracy:.2%}")

### What just happened?

The model correctly predicted heart disease (or its absence) for about **80% of patients** in the test set. Not bad for our first classifier!

Notice we used `X_train_scaled` — k-NN needs scaled features because it calculates distances. Without scaling, cholesterol (values up to 564) would overpower exercise_angina (just 0 or 1).

### 🔄 Your Turn

In the empty cell below, copy the k-NN code from above and change `n_neighbors=5` to `n_neighbors=3`. Run it.

- Did accuracy go up or down?
- Try `n_neighbors=1` — what happens?
- Try `n_neighbors=15` — better or worse?

In [ ]:
# YOUR TURN: Try different K values



---
## Example 7.2: Decision Tree + Testing Multiple K Values

Now we'll train a completely different algorithm — a decision tree that asks yes/no questions about features. Then we'll systematically test multiple K values for k-NN to find the sweet spot.

In [ ]:
# ============================================
# Example 7.2: Decision Tree + K Value Comparison
# Purpose: Train decision tree, compare K values
# ============================================

# --- Part A: Train a Decision Tree ---

# Note: using UNscaled data — trees don't need scaling
tree = DecisionTreeClassifier(max_depth=4, random_state=42)
tree.fit(X_train, y_train)

tree_predictions = tree.predict(X_test)
tree_accuracy = accuracy_score(y_test, tree_predictions)
print(f"Decision Tree Accuracy: {tree_accuracy:.2%}")

# --- Part B: Test Multiple K Values ---

k_values = [1, 3, 5, 7, 9, 11]
k_accuracies = []

for k in k_values:
    knn_temp = KNeighborsClassifier(n_neighbors=k)
    knn_temp.fit(X_train_scaled, y_train)
    acc = accuracy_score(y_test, knn_temp.predict(X_test_scaled))
    k_accuracies.append(acc)
    print(f"k-NN Accuracy (K={k}): {acc:.2%}")

# --- Part C: Plot K vs. Accuracy ---

plt.figure(figsize=(8, 4))
plt.plot(k_values, k_accuracies, marker='o', color='coral', linewidth=2)
plt.xlabel('K (Number of Neighbors)')
plt.ylabel('Accuracy')
plt.title('k-NN: How K Affects Accuracy')
plt.xticks(k_values)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### What just happened?

Two key observations:

1. **K=1 performs worst** (73.77%) — it's overfitting to individual noisy points. **K=3 peaks** at 81.97%, then accuracy drifts down as K grows.
2. **The decision tree and k-NN (K=5) tied** at 80.33%. That doesn't mean they make the same predictions — they could be getting different patients right and wrong.

The decision tree used **unscaled** data (`X_train` not `X_train_scaled`) because trees split on thresholds, not distances.

### 🔄 Your Turn

In the empty cell below:

1. Change `max_depth` on the decision tree from 4 to 2. Does accuracy change? (Hint: it might surprise you.)
2. Add K=15 and K=21 to the `k_values` list. Does the downward trend continue?

In [ ]:
# YOUR TURN: Experiment with max_depth and more K values



---
## Example 7.3: Full Classification Pipeline — Head-to-Head Comparison

Now we bring it all together: train both classifiers with our best settings, compare them with classification reports, and visualize the decision tree to see *why* it makes its predictions.

In [ ]:
# ============================================
# Example 7.3: Full Classification Comparison
# Purpose: Complete pipeline comparing k-NN
#          and decision tree on heart disease data
# ============================================

# --- Step 1: Train both models with best settings ---

# k-NN with best K from our experiment
best_k = 3
knn_final = KNeighborsClassifier(n_neighbors=best_k)
knn_final.fit(X_train_scaled, y_train)

# Decision tree with controlled depth
tree_final = DecisionTreeClassifier(max_depth=4, random_state=42)
tree_final.fit(X_train, y_train)

# --- Step 2: Generate predictions ---
knn_preds = knn_final.predict(X_test_scaled)
tree_preds = tree_final.predict(X_test)

# --- Step 3: Classification reports ---
print("=" * 50)
print(f"k-NN Classification Report (K={best_k})")
print("=" * 50)
print(classification_report(y_test, knn_preds,
      target_names=['No Disease', 'Heart Disease']))

print("=" * 50)
print("Decision Tree Classification Report")
print("=" * 50)
print(classification_report(y_test, tree_preds,
      target_names=['No Disease', 'Heart Disease']))

### Your Turn: Build the comparison table and visualize the tree

Now it's your turn to finish the pipeline. Complete the two cells below.

In [ ]:
# --- Step 4: Build a comparison table ---
# YOUR CODE HERE
# Create a DataFrame comparing the two models
# Columns: Algorithm, Accuracy, Explainable?, Needs Scaling?
# Hint: use accuracy_score(y_test, knn_preds) and accuracy_score(y_test, tree_preds)

knn_acc = accuracy_score(y_test, knn_preds)
tree_acc = accuracy_score(y_test, tree_preds)

# YOUR CODE HERE — build the comparison DataFrame and print it



In [ ]:
# --- Step 5: Visualize the decision tree ---
# YOUR CODE HERE
# Use plot_tree() to visualize tree_final
# Parameters to include:
#   feature_names=X.columns.tolist()
#   class_names=['No Disease', 'Heart Disease']
#   filled=True, rounded=True, fontsize=9
# Set figure size to (14, 7)



In [ ]:
# --- Step 6: Feature importances (what does the tree value most?) ---
print("\n🔍 Feature Importances (Decision Tree):")
for name, imp in zip(X.columns, tree_final.feature_importances_):
    print(f"  {name}: {imp:.3f}")

### Interpretation

The results tell a clear story:

- **k-NN (K=3) wins on accuracy:** 81.97% vs. 80.33% — a close race
- **Decision tree wins on explainability:** You can read the tree and trace every decision. The tree reveals that `chest_pain_type` is the most important feature (0.388), followed by `age` (0.134) and `sex` (0.131)
- **Neither is universally better.** The right choice depends on what you value: accuracy or transparency

This is the core professional skill: comparing algorithms and making reasoned choices.

---
## What We Built

In this guided project, you:

1. Trained your first k-NN classifier and achieved 80%+ accuracy on real patient data
2. Experimented with different K values and found the sweet spot at K=3
3. Trained a decision tree and visualized its reasoning
4. Compared both algorithms side by side with classification reports
5. Learned that `chest_pain_type` is the most important feature for predicting heart disease in this dataset

**Up next in Chapter 8:** We add two more classifiers — linear classifiers and support vector machines — then compare all four side by side. The four-classifier comparison is where things get really interesting.

---

**Keep this notebook for reference.** You'll use these same skills in your NG07 assignment, where you'll explore this dataset further with different settings and analyze its ethical implications.